# Вводный модуль · Данные — основа всего
**Практикум.** Почему одна и та же модель у одних работает, а у других нет? Проверим своими руками — и посчитаем,
сколько памяти нужно модели, ещё до того, как её скачать.

**Как работать.** Меню **«Среда выполнения → Выполнить все»** (Runtime → Run all) — и читайте сверху вниз.
В ячейках с полями справа меняйте значения и запускайте ячейку заново (▶ слева или Ctrl+Enter). Сломать ничего
нельзя: **«Среда выполнения → Отключиться и удалить среду»** вернёт всё как было. Код можно не читать — но если
интересно, он открыт и короткий.

Данные — набор рукописных цифр из scikit-learn (1 797 изображений 8 × 8 пикселей).

## 1. Одна модель — «лаборатория» и «поле»
Учим простую модель на ровных, чистых цифрах и проверяем на таких же. Потом — на тех же цифрах, но чуть
повёрнутых, сдвинутых, тусклых и с шумом: так выглядит «поле» по сравнению с «лабораторией».

In [ ]:
#@title Обучаем модель на «лабораторных» данных
import numpy as np, matplotlib.pyplot as plt, warnings
from scipy import ndimage as ndi
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

X, y = load_digits(return_X_y=True)
imgs = X.reshape(-1, 8, 8)
Xtr, Xte, ytr, yte, Itr, Ite = train_test_split(X, y, imgs, test_size=0.4, random_state=0, stratify=y)
model = LogisticRegression(max_iter=3000).fit(Xtr / 16, ytr)
acc_lab = (model.predict(Xte / 16) == yte).mean()
print(f'Обучали на {len(ytr)} цифрах, проверяем на {len(yte)} других.')
print(f'Точность на таких же «лабораторных» данных: {acc_lab:.1%}')

In [ ]:
#@title Переносим в «поле»: меняйте условия съёмки
max_angle = 15  #@param {type:"slider", min:0, max:45, step:1}
max_shift = 1.0  #@param {type:"slider", min:0, max:2, step:0.1}
min_brightness = 0.6  #@param {type:"slider", min:0.2, max:1.0, step:0.05}
noise = 1.5  #@param {type:"slider", min:0, max:4, step:0.1}

def field(im, seed):
    r = np.random.default_rng(seed)
    out = ndi.rotate(im, r.uniform(-max_angle, max_angle), reshape=False, order=1)
    out = ndi.shift(out, r.uniform(-max_shift, max_shift, 2), order=1)
    out = out * r.uniform(min_brightness, 1.0) + r.normal(0, noise, out.shape)
    return np.clip(out, 0, 16)

Fte = np.stack([field(im, i) for i, im in enumerate(Ite)])
acc_field = (model.predict(Fte.reshape(len(Fte), -1) / 16) == yte).mean()
fig, axs = plt.subplots(2, 8, figsize=(12, 3.4))
for i in range(8):
    axs[0, i].imshow(Ite[i], cmap='Greys'); axs[1, i].imshow(Fte[i], cmap='Greys')
    axs[0, i].axis('off'); axs[1, i].axis('off')
axs[0, 0].set_title('лаборатория', loc='left', fontsize=10); axs[1, 0].set_title('поле', loc='left', fontsize=10)
plt.show()
print(f'Лаборатория: {acc_lab:.1%}   ·   поле: {acc_field:.1%}')

**Что заметить.** При значениях по умолчанию точность падает с 97 % до 27–28 % — модель та же, изменились данные.

**Задание 1.** Обнулите все условия, кроме одного (например, оставьте только поворот). Какое условие съёмки бьёт
сильнее всего? Начиная с какого угла поворота точность падает ниже 80 %?

In [ ]:
#@title Лечим данными, а не моделью: добавим «полевые» примеры в обучение
copies = 3  #@param {type:"slider", min:0, max:6, step:1}
Xaug = [Xtr / 16]; yaug = [ytr]
for c in range(copies):
    Xaug.append(np.stack([field(im, 10_000 * (c + 1) + i) for i, im in enumerate(Itr)]).reshape(len(Itr), -1) / 16)
    yaug.append(ytr)
model2 = LogisticRegression(max_iter=3000).fit(np.vstack(Xaug), np.concatenate(yaug))
print(f'Та же модель, обучение + {copies} «полевые» копии каждого примера:')
print(f'  лаборатория: {(model2.predict(Xte / 16) == yte).mean():.1%}')
print(f'  поле:        {(model2.predict(Fte.reshape(len(Fte), -1) / 16) == yte).mean():.1%}')

**Что заметить.** Архитектура не менялась — менялись только данные, на которых учили. Об этом весь курс:
модель можно скачать, а данные под свою задачу придётся собрать.

## 2. Сколько памяти нужно модели
Память под веса = **число параметров × байты на один параметр**. Сверх весов модели нужна ещё память на работу
(для языковых моделей — на контекст), поэтому берите запас.

In [ ]:
#@title Калькулятор памяти
params_billion = 11.5  #@param {type:"number"}
my_memory_gb = 16  #@param {type:"number"}
reserve = 1.2  #@param {type:"slider", min:1.0, max:2.0, step:0.05}

rows = [('FP32', 4), ('FP16 / BF16', 2), ('INT8', 1), ('INT4 (Q4)', 0.5)]
print(f'Модель на {params_billion} млрд параметров; у вас {my_memory_gb} ГБ; запас ×{reserve}\n')
print(f'{"формат":<13}{"веса, ГБ":>10}{"с запасом":>12}   влезет?')
for name, b in rows:
    w = params_billion * 1e9 * b / 1024**3
    print(f'{name:<13}{w:>10.1f}{w * reserve:>12.1f}   {"да" if w * reserve <= my_memory_gb else "нет"}')

**Задание 2.** Откройте карточку любой модели на Hugging Face (QR в презентации), найдите число параметров и
посчитайте: на каком формате она влезет в ваш ноутбук? А в видеокарту на 24 ГБ? Как сжимать без потери качества и
как это проверять — в модуле 1.

---
*© 2026 Наталия Тылевич. Авторский учебный материал курса «Прикладное ML: от данных к системе». Можно выполнять и хранить личную копию для заданий курса; распространять и использовать в других курсах — только с согласия автора. Условия — `LICENSE.md` в репозитории.*